# Milestone 1

## QA Pipeline

In [78]:
import nltk
import re
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abdel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abdel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load Dataset

In [79]:
import os

path = "data/QA"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        print(f, len(text))

0urc3PabvOs_QA.csv 39181
3AwL93uolIA_QA.csv 34114
8rcY34IA-6I_QA.csv 34706
8ZL2AAxQmLQ_QA.csv 42284
ArytJ_HZ-1E_QA.csv 42329
DbuoaakWh7g_QA.csv 35242
gv5hDK2YQfA_QA.csv 33161
IxsJEffQAZA_QA.csv 35730
MOEwXtL2DQ4_QA.csv 37415
nc8oJTETqoI_QA.csv 37153
NJ-JcDcff8o_QA.csv 38688
yc7x5jNhXIQ_QA.csv 34614
z2NGnjXG5uQ_QA.csv 40717


### Inspect the Dataset

In [80]:
import pandas as pd

df = pd.read_csv("data/QA/0urc3PabvOs_QA.csv") # Path to the first QA file
df.head()

,video_id,video_title,question_id,question,answer,difficulty
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,ماذا ورد في هذا الموضع من النص؟,"افتح موضوع جديد يا ""ميدو""، أنا مش ناقص!",Easy
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,ما الجملة المذكورة في هذا السياق؟,وبعد كدا، هتنطفي هي كمان.,Medium
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,كيف صيغت العبارة هنا؟,- أيوة.,Easy
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,ما الذي قيل في هذا الجزء؟,- عشان كُل حاجة بتنتهي.,Medium
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,ما النص الحرفي المذكور في هذه الفقرة؟,- عشان الفيزيا بتقول كدا.,Easy


In [81]:
df["question_length"] = df["question"].apply(len)
df["answer_length"] = df["answer"].apply(len)

print(df.describe())

       question_length  answer_length
count       300.000000     300.000000
mean         29.400000      26.383333
std           5.722689       7.685045
min          21.000000       5.000000
25%          25.000000      21.000000
50%          31.000000      26.000000
75%          33.000000      32.000000
max          37.000000      45.000000


In [82]:
from collections import Counter

path = "data/QA"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        words = text.split()
        freq = Counter(words)
        
        print(freq.most_common(20))

[('و', 600), ('كل', 304), ('الشمس', 302), ('شيء', 301), ('0urc3PabvOs,مصير', 300), ('الأرض', 300), ('|', 300), ('في', 278), ('هذا', 186), ('من', 79), ('هذه', 67), ('الذي', 61), ('ورد', 60), ('الموضع', 60), ('الجملة', 60), ('المذكورة', 60), ('صيغت', 60), ('العبارة', 60), ('قيل', 60), ('النص', 60)]
[('3AwL93uolIA,الساموراي', 300), ('|', 300), ('في', 276), ('هذا', 127), ('النص', 120), ('هذه', 62), ('ورد', 60), ('حول', 60), ('الجملة', 60), ('المذكورة', 60), ('صيغت', 60), ('العبارة', 60), ('الذي', 60), ('قيل', 60), ('الحرفي', 60), ('المذكور', 60), ('من', 20), ('على', 12), ('كان', 10), ('أو', 10)]
[('في', 608), ('8rcY34IA-6I,الأخطبوط', 290), ('|', 290), ('الذي', 290), ('ورد', 290), ('النص', 290), ('الفقرة', 290), ('رقم', 290), ('ما', 30), ('على', 18), ('من', 14), ('الأخطبوط', 12), ('هو', 10), ('اللي', 8), ('أو', 8), ('إن', 7), ('غير', 7), ('عنده', 7), ('دراع', 7), ('انت', 5)]
[('قبل', 303), ('جبل', 302), ('30', 301), ('طن', 301), ('8ZL2AAxQmLQ,كيف', 300), ('تنقل', 300), ('وزنه', 300), ('أن',

### Clean And normalize Text Fields

In [83]:
def process_text_column(df, column_name):

    df[column_name] =df[column_name].apply(remove_stopwords_and_punctuation_from_text)
    df[column_name] =df[column_name].apply(normalize_text)
    return df

def remove_stopwords_and_punctuation_from_text(text):
    nltk_stop_words = set(stopwords.words('arabic'))

    # Tokenize the text
    tokens = nltk.word_tokenize(text)

    # punctuation ='!"$%&()*,-./:;<=>?@[\\]^_`{|}~'
    punctuation = [
    "،","؛","؟","ـ","«","»","‹","›","“","”","‘","’",
    ".",",",";",":","!","?","-","_","(",")","[","]","{","}",
    "\"","'","/","\\","|","@","#","$","%","^","&","*","+","=","<",">","~","`","``","''"
    ]
    for p in punctuation:
        text = text.replace(p, '')

    tokens = nltk.word_tokenize(text)

    filtered_tokens = [word for word in tokens if word not in nltk_stop_words]
    # Join the tokens back into a string
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text



def normalize_text(text):
    
    # 1. Convert English letters to lowercase
    text = text.lower()
    
    # 2. Remove Arabic Tashkeel (diacritics)
    tashkeel = r'[\u0617-\u061A\u064B-\u0652]'
    text = re.sub(tashkeel, '', text)

    # 3. Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    # text = re.sub("ؤ", "و", text)
    # text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("گ", "ك", text)

    # 4. Remove Tatweel (ـ)
    text = re.sub("ـ", "", text)

    return text

In [84]:
# print("ماذا" in set(stopwords.words('arabic')))

In [85]:
process_text_column(df, "question")

,video_id,video_title,question_id,question,answer,difficulty,question_length,answer_length
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,الموضع النص,"افتح موضوع جديد يا ""ميدو""، أنا مش ناقص!",Easy,31,39
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,الجمله المذكوره السياق,وبعد كدا، هتنطفي هي كمان.,Medium,33,25
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,صيغت العباره,- أيوة.,Easy,21,7
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,قيل الجزء,- عشان كُل حاجة بتنتهي.,Medium,25,23
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,النص الحرفي المذكور الفقره,- عشان الفيزيا بتقول كدا.,Easy,37,25
...,...,...,...,...,...,...,...,...
295,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q296,الموضع النص,"قاعدين في ظروف عادية، اوعى تتحلل!""",Medium,31,34
296,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q297,الجمله المذكوره السياق,إلى 10 دوتشيليون سنة،,Easy,33,21
297,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q298,صيغت العباره,كل حاجة،,Medium,21,8
298,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q299,قيل الجزء,هيتحلل هو كمان.,Easy,25,15


In [86]:
process_text_column(df, "answer")

,video_id,video_title,question_id,question,answer,difficulty,question_length,answer_length
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,الموضع النص,افتح موضوع جديد ميدو مش ناقص,Easy,31,39
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,الجمله المذكوره السياق,وبعد كدا هتنطفي كمان,Medium,33,25
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,صيغت العباره,ايوه,Easy,21,7
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,قيل الجزء,عشان كل حاجه بتنتهي,Medium,25,23
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,النص الحرفي المذكور الفقره,عشان الفيزيا بتقول كدا,Easy,37,25
...,...,...,...,...,...,...,...,...
295,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q296,الموضع النص,قاعدين ظروف عاديه اوعي تتحلل,Medium,31,34
296,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q297,الجمله المذكوره السياق,10 دوتشيليون سنه,Easy,33,21
297,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q298,صيغت العباره,حاجه,Medium,21,8
298,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q299,قيل الجزء,هيتحلل كمان,Easy,25,15


In [ ]:
# Save the processed DataFrame to a new CSV file
# df.to_csv("output.csv", index=False, encoding="utf-8-sig")

## Transcripts Pipeline

### Load Dataset

In [87]:
import os

path = "data/Transcripts"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        print(f, len(text))

أعظم طائرة حربية  الدحيح.txt 38222
الأخطبوط  الدحيح.txt 38054
الساموراي  الدحيح.txt 34077
تاج محل  الدحيح.txt 26431
جون كينيدي  الدحيح.txt 56962
فيزياء و فلسفة الحركة  الدحيح.txt 47095
كيف تحولت روسيا إلى إمبراطورية؟  الدحيح.txt 59094
كيف تسيطر على عقول البشر؟  الدحيح.txt 45570
كيف تنقل جبل وزنه 30 طن قبل أن يغرق؟  الدحيح.txt 38574
مصير الأرض و الشمس و كل شيء  الدحيح.txt 33909
معركة ذي قار  الدحيح.txt 51260
منابع النيل  الدحيح.txt 49836
هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح.txt 35210


### Inspect the Dataset

In [88]:
from collections import Counter

path = "data/Transcripts"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        words = text.split()
        freq = Counter(words)
        
        print(freq.most_common(20))

[('يا', 142), ('في', 128), ('عزيزي،', 120), ('اللي', 102), ('على', 90), ('من', 88), ('ما', 83), ('إن', 67), ('دا', 45), ('مش', 41), ('دي', 36), ('هو', 36), ('انت', 33), ('طيارة', 32), ('الطيارة', 30), ('عشان', 29), ('كل', 29), ('كان', 28), ('احنا', 23), ('الـF-35', 21)]
[('يا', 134), ('في', 122), ('عزيزي،', 94), ('ما', 85), ('إن', 82), ('من', 82), ('على', 68), ('الأخطبوط', 65), ('اللي', 64), ('هو', 44), ('دا', 42), ('زي', 39), ('مش', 38), ('كدا،', 34), ('انت', 31), ('كل', 31), ('عشان', 31), ('بس', 28), ('لمّا', 26), ('فيه', 25)]
[('يا', 124), ('في', 113), ('عزيزي،', 110), ('كان', 65), ('من', 63), ('اللي', 63), ('الساموراي', 57), ('إن', 56), ('ما', 49), ('على', 33), ('كانوا', 31), ('دا', 30), ('أو', 29), ('كانت', 29), ('عشان', 28), ('زي', 25), ('مش', 24), ('هُما', 21), ('كدا،', 21), ('هو', 19)]
[('في', 96), ('يا', 91), ('عزيزي،', 67), ('من', 66), ('اللي', 64), ('ما', 50), ('على', 47), ('إن', 39), ('دا', 32), ('مش', 31), ('كان', 28), ('كل', 24), ('زي', 22), ('دي', 22), ('"تاج', 21), ('"ش

### Clean And normalize Text Fields

In [91]:
def process_text_transcript(text):
    text = remove_timestamps(text)
    text = remove_stopwords_and_punctuation_from_text(text)
    text = normalize_text(text)
    return text

def remove_timestamps(text):
    return re.sub(r'\d+\.\d+:', '', text)

In [95]:
file_path = "data/Transcripts/أعظم طائرة حربية  الدحيح.txt"
with open(file_path, encoding="utf-8") as file:
    text = file.read()
    processed_text = process_text_transcript(text)
    print(processed_text[:500])  # Print the first 500 characters of the processed text

سياده الكولونيل صبرك محله مبروك علينا عملنا افجر طياره تاريخ امريكا متحمس جدا امبارح وريني اقدم لحضرتك فخر الطيران الامريكي الf35 دي شكلها اللي احنا عملناها كدا التكنولوجيا اللي سابقه بسنين ضوئيه حضرتك يعني سنين ضوئيه مش مهم مش مهم احكيلي عنها كدا القطعه الفنيه اللي حضرتك دي اقوي كمبيوتر كمبيونر نس معلش يعني سؤال ساذج حته تانيه بتقتلها تقريبا كام طفل الدقيقه مش عارف فندم بصراحه كام طفل واقفين يعني اخترعنهاش عشان تtarget اطفال بالتحديد يعني طب معلش يعني سؤال ساذج تاني يعني افترضنا شخص ومثلا مثلا 


In [ ]:
# Save processed text
# output_path = "processed_transcript.txt"
# with open(output_path, "w", encoding="utf-8") as file:
#     file.write(processed_text)